In [ ]:
# LUARAN LUAS AREA PREDIKSI POTENTIAL PER KOTA (m2) + PETA PER KOTA

# Impor, path, dan pengaturan
import os, zipfile, glob
import numpy as np
import pandas as pd
import torch, torch.nn as nn
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from google.colab import drive

drive.mount('/content/drive')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Path
CSV_META   = '/content/drive/MyDrive/SolarPanel_Preprocessing/parameter_mentah_per_tile.csv'
ZIP_TILES  = '/content/drive/MyDrive/SolarPanel_Preprocessing/AllTiles_backup.zip'
SPLIT_DIR  = '/content/drive/MyDrive/SolarPanel_Label/Dataset_Split_Final_aug'
MODEL_PATH = '/content/drive/MyDrive/SolarPanel_Training/Model_Terbaik/model_best_EfficientNetB0.pth.tar'
FOLDER_GEE = '/content/drive/MyDrive/SolarPanel_Dataset_2024'   # GeoTIFF S2_RGB_{kota}.tif
OUT_DIR    = '/content/drive/MyDrive/SolarPanel_Training/Hasil_Luas_Area'
TILE_DIR   = '/content/AllTiles_local'                          # hasil ekstrak zip (lokal)
os.makedirs(OUT_DIR, exist_ok=True)

UKURAN_TILE = 224
SKALA_M     = 10                      # resolusi nominal Sentinel-2 (m)
AREA_NOMINAL_M2 = (UKURAN_TILE * SKALA_M) ** 2   # 5.017.600 m2 per patch
print('Luas nominal 1 patch (m2):', AREA_NOMINAL_M2)

Mounted at /content/drive
Luas nominal 1 patch (m2): 5017600


In [ ]:
# [2] Ekstrak semua patch (lokal, bukan ke Drive) dan baca metadata
if not os.path.isdir(TILE_DIR) or len(os.listdir(TILE_DIR)) == 0:
    os.makedirs(TILE_DIR, exist_ok=True)
    with zipfile.ZipFile(ZIP_TILES) as z:
        z.extractall(TILE_DIR)

meta = pd.read_csv(CSV_META)
print('Jumlah patch pada metadata :', len(meta))
print('Jumlah file PNG hasil ekstrak:', len(glob.glob(os.path.join(TILE_DIR, '*.png'))))
print(meta[['filename', 'kota', 'baris', 'kolom']].head())

Jumlah patch pada metadata : 4705
Jumlah file PNG hasil ekstrak: 4705
              filename       kota  baris  kolom
0  BandaAceh_r0_c0.png  BandaAceh      0      0
1  BandaAceh_r0_c1.png  BandaAceh      0      1
2  BandaAceh_r0_c2.png  BandaAceh      0      2
3  BandaAceh_r0_c3.png  BandaAceh      0      3
4  BandaAceh_r0_c4.png  BandaAceh      0      4


In [ ]:
# Tentukan status tiap patch: train / val / test / ambigu
# Label referensi diambil dari STRUKTUR FOLDER split (bukan dihitung ulang).
# Patch yang tidak ada di folder split mana pun = Ambigu (atau dibuang).
status = {}
for split in ['train', 'val', 'test']:
    for kelas in ['Potential', 'Not_Potential']:
        for p in glob.glob(os.path.join(SPLIT_DIR, split, kelas, '*.png')):
            nama = os.path.basename(p)
            status[nama] = (split, kelas)          # nama hasil augmentasi tidak ada di meta -> diabaikan

meta['split'] = meta['filename'].map(lambda n: status.get(n, (None, None))[0])
meta['label_ref'] = meta['filename'].map(lambda n: status.get(n, (None, None))[1])
meta['split'] = meta['split'].fillna('ambigu_atau_tidak_terpakai')
meta['label_ref'] = meta['label_ref'].fillna('Ambigu')

print('\nJumlah patch per status:')
print(meta['split'].value_counts())
# Cek kesesuaian dengan laporan
print('Test =', (meta['split'] == 'test').sum(), '(laporan: 449)')


Jumlah patch per status:
split
train                         3150
val                            906
test                           449
ambigu_atau_tidak_terpakai     200
Name: count, dtype: int64
Test = 449 (laporan: 449)


In [ ]:
# Muat model terbaik
model = models.efficientnet_b0(weights=None)
model.classifier[0].p = 0.5
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)
ckpt = torch.load(MODEL_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt['state_dict'])
model = model.to(device).eval()

tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class TileDS(Dataset):
    def __init__(self, names): self.names = list(names)
    def __len__(self): return len(self.names)
    def __getitem__(self, i):
        img = Image.open(os.path.join(TILE_DIR, self.names[i])).convert('RGB')
        return tf(img)

loader = DataLoader(TileDS(meta['filename']), batch_size=64, shuffle=False, num_workers=0)

In [ ]:
# Inferensi pada SEMUA patch (kelas 0 = Not_Potential, 1 = Potential)
probs = []
with torch.no_grad():
    for x in loader:
        p = torch.softmax(model(x.to(device)), dim=1).cpu().numpy()
        probs.append(p)
probs = np.vstack(probs)
meta['prob_not_potential'] = probs[:, 0]
meta['prob_potential']     = probs[:, 1]
meta['prediksi'] = np.where(probs[:, 1] >= probs[:, 0], 'Potential', 'Not_Potential')

# Verifikasi
t = meta[meta['split'] == 'test']
print(pd.crosstab(t['label_ref'], t['prediksi']))

prediksi       Not_Potential  Potential
label_ref                              
Not_Potential            290         13
Potential                 14        132


In [ ]:
# Luas tiap patch (m2)
# Nominal: 224 x 10 m = 2.240 m per sisi. Raster diekspor di EPSG:4326 dengan skala 10 m,
# sehingga luas sebenarnya sedikit lebih kecil di luar ekuator (faktor cos(lintang)).
# Di bawah dihitung luas yang dikoreksi memakai transform GeoTIFF tiap kota.
import rasterio
luas = []
for kota, g in meta.groupby('kota'):
    path_tif = os.path.join(FOLDER_GEE, f'S2_RGB_{kota}.tif')
    with rasterio.open(path_tif) as src:
        T = src.transform
        for idx, r in g.iterrows():
            x0, y0 = T * (r['kolom'] * UKURAN_TILE, r['baris'] * UKURAN_TILE)
            x1, y1 = T * ((r['kolom'] + 1) * UKURAN_TILE, (r['baris'] + 1) * UKURAN_TILE)
            lat_c = (y0 + y1) / 2
            lebar_m  = abs(x1 - x0) * 111320 * np.cos(np.radians(lat_c))
            tinggi_m = abs(y1 - y0) * 110574
            luas.append((idx, lebar_m * tinggi_m, (x0 + x1) / 2, lat_c))
for idx, a, lon, lat in luas:
    meta.loc[idx, 'luas_m2'] = a
    meta.loc[idx, 'lon_tengah'] = lon
    meta.loc[idx, 'lat_tengah'] = lat
print(meta['luas_m2'].describe())

meta.to_csv(os.path.join(OUT_DIR, 'prediksi_semua_patch.csv'), index=False)

count    4.705000e+03
mean     4.947450e+06
std      1.223403e+04
min      4.926511e+06
25%      4.937817e+06
50%      4.946557e+06
75%      4.953741e+06
max      4.984021e+06
Name: luas_m2, dtype: float64


In [ ]:
# Tabel ringkasan per kota
def ringkas(df):
    return pd.Series({
        'jumlah_patch'        : len(df),
        'prediksi_potential'  : (df['prediksi'] == 'Potential').sum(),
        'luas_potential_km2'  : df.loc[df['prediksi'] == 'Potential', 'luas_m2'].sum() / 1e6,
        'luas_potential_m2'   : df.loc[df['prediksi'] == 'Potential', 'luas_m2'].sum(),
        'luas_total_km2'      : df['luas_m2'].sum() / 1e6,
    })

semua = meta.groupby('kota').apply(ringkas)
semua['persen_potential'] = 100 * semua['luas_potential_km2'] / semua['luas_total_km2']
print('\n=== SEMUA patch (termasuk yang dipakai melatih model) ===')
print(semua.round(2))

uji = meta[meta['split'] == 'test'].groupby('kota').apply(ringkas)
print('\n=== HANYA data uji (patch yang tidak dipakai melatih) ===')
print(uji.round(2))

# Perbandingan dengan label referensi (hanya patch berlabel), sebagai pemeriksaan
lab = meta[meta['label_ref'].isin(['Potential', 'Not_Potential'])]
ref = lab[lab['label_ref'] == 'Potential'].groupby('kota')['luas_m2'].sum() / 1e6
semua['luas_potential_label_ref_km2'] = ref
semua.to_csv(os.path.join(OUT_DIR, 'ringkasan_luas_per_kota_semua_patch.csv'))
uji.to_csv(os.path.join(OUT_DIR, 'ringkasan_luas_per_kota_data_uji.csv'))

/tmp/ipykernel_2256/3782691055.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  semua = meta.groupby('kota').apply(ringkas)
/tmp/ipykernel_2256/3782691055.py:16: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  uji = meta[meta['split'] == 'test'].groupby('kota').apply(ringkas)



=== SEMUA patch (termasuk yang dipakai melatih model) ===
            jumlah_patch  prediksi_potential  luas_potential_km2  \
kota                                                               
BandaAceh          117.0                37.0              183.55   
Bandung            513.0               169.0              836.11   
Denpasar           304.0               100.0              492.81   
Jakarta           1426.0               717.0             3551.86   
Jayapura            30.0                 6.0               29.87   
Kediri             532.0               228.0             1125.22   
Makassar           285.0                73.0              362.36   
Pontianak          192.0                33.0              164.47   
Surabaya           624.0               237.0             1171.31   
Yogyakarta         682.0               291.0             1436.99   

            luas_potential_m2  luas_total_km2  persen_potential  
kota                                                      

In [ ]:
# Peta di atas citra asli per kota: kiri citra mentah, kanan citra + prediksi
import rasterio
from matplotlib.patches import Rectangle, Patch

DECIMATE = 4   # naikkan (6 atau 8) untuk kota besar seperti Jakarta bila memori penuh

def stretch_kontras(tile_rgb, vmin=0.01, vmax=0.28, gamma=1.3):
    img = np.clip(tile_rgb, vmin, vmax)
    img = (img - vmin) / (vmax - vmin)
    return np.power(img, 1.0 / gamma)

warna_overlay = {
    'Potential': '#2563EB',      # Biru
    'Not_Potential': '#EA580C'   # Oranye
}

for kota, g in meta.groupby('kota'):
    with rasterio.open(os.path.join(FOLDER_GEE, f'S2_RGB_{kota}.tif')) as src:
        H, W = src.height, src.width
        arr = src.read([1, 2, 3], out_shape=(3, H // DECIMATE, W // DECIMATE))
        T = src.transform
    rgb = stretch_kontras(np.nan_to_num(np.transpose(arr, (1, 2, 0)).astype('float32')))

    x_kiri, y_atas = T * (0, 0)
    x_kanan, y_bawah = T * (W, H)
    extent = [x_kiri, x_kanan, y_bawah, y_atas]

    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    for ax in axes:
        ax.imshow(rgb, extent=extent, aspect='auto')
        ax.set_xlabel('Bujur'); ax.set_ylabel('Lintang')
    axes[0].set_title(f'{kota}: citra Sentinel-2 RGB')
    axes[1].set_title(f'{kota}: prediksi EfficientNet-B0 per patch')

    for _, r in g.iterrows():
        x0, y0 = T * (r['kolom'] * UKURAN_TILE, r['baris'] * UKURAN_TILE)
        x1, y1 = T * ((r['kolom'] + 1) * UKURAN_TILE, (r['baris'] + 1) * UKURAN_TILE)
        ambigu = (r['label_ref'] == 'Ambigu')
        axes[1].add_patch(Rectangle((min(x0, x1), min(y0, y1)), abs(x1 - x0), abs(y1 - y0),
                                    facecolor=warna_overlay[r['prediksi']], alpha=0.35,
                                    edgecolor='yellow' if ambigu else 'white',
                                    linewidth=1.5 if ambigu else 0.3,
                                    hatch='//' if ambigu else None))

        # Legenda di luar area citra, di bagian bawah
        legend_handles = [
            Patch(
                color=warna_overlay['Potential'],
                alpha=0.6,
                label='Potential'
            ),
            Patch(
                color=warna_overlay['Not_Potential'],
                alpha=0.6,
                label='Not_Potential'
            ),
            Patch(
                facecolor='none',
                edgecolor='yellow',
                hatch='//',
                label='Ambigu (di luar pelatihan)'
            )
        ]

        fig.legend(
            handles=legend_handles,
            loc='lower center',
            bbox_to_anchor=(0.5, 0.02),
            ncol=3,
            fontsize=10,
            frameon=True
        )

        # Sisakan ruang di bawah gambar untuk legenda
        fig.subplots_adjust(bottom=0.20, wspace=0.25)

    plt.savefig(
        os.path.join(OUT_DIR, f'peta_citra_{kota}.png'),
        dpi=200,
        bbox_inches='tight'
    )
    plt.close()
print('Peta di atas citra asli tersimpan di', OUT_DIR)

Peta di atas citra asli tersimpan di /content/drive/MyDrive/SolarPanel_Training/Hasil_Luas_Area


In [ ]:

# PETA HASIL PREDIKSI SAJA - UNTUK GAMBAR 4.7

import os
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from matplotlib.patches import Rectangle, Patch

warna_overlay = {
    "Potential": "#2563EB",
    "Not_Potential": "#EA580C"
}

for kota, g in meta.groupby("kota"):

    path_tif = os.path.join(
        FOLDER_GEE, f"S2_RGB_{kota}.tif"
    )

    with rasterio.open(path_tif) as src:
        H, W = src.height, src.width
        T = src.transform

        arr = src.read(
            [1, 2, 3],
            out_shape=(3, H // 4, W // 4)
        )

    rgb = stretch_kontras(
        np.nan_to_num(
            np.transpose(arr, (1, 2, 0)).astype("float32")
        )
    )

    x_kiri, y_atas = T * (0, 0)
    x_kanan, y_bawah = T * (W, H)

    fig, ax = plt.subplots(figsize=(7, 7))

    ax.imshow(
        rgb,
        extent=[x_kiri, x_kanan, y_bawah, y_atas],
        aspect="auto"
    )

    for _, r in g.iterrows():
        x0, y0 = T * (
            r["kolom"] * UKURAN_TILE,
            r["baris"] * UKURAN_TILE
        )

        x1, y1 = T * (
            (r["kolom"] + 1) * UKURAN_TILE,
            (r["baris"] + 1) * UKURAN_TILE
        )

        ambigu = r["label_ref"] == "Ambigu"

        ax.add_patch(
            Rectangle(
                (min(x0, x1), min(y0, y1)),
                abs(x1 - x0),
                abs(y1 - y0),
                facecolor=warna_overlay[r["prediksi"]],
                alpha=0.65,
                edgecolor="yellow" if ambigu else "white",
                linewidth=1.5 if ambigu else 0.3,
                hatch="//" if ambigu else None
            )
        )

    ax.set_title(f"Wilayah Studi {kota}")
    ax.set_xlabel("Bujur")
    ax.set_ylabel("Lintang")

    legend_handles = [
        Patch(
            facecolor=warna_overlay["Potential"],
            label="Potential"
        ),
        Patch(
            facecolor=warna_overlay["Not_Potential"],
            label="Not_Potential"
        ),
        Patch(
            facecolor="none",
            edgecolor="yellow",
            hatch="//",
            label="Ambigu"
        )
    ]

    fig.legend(
        handles=legend_handles,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.02),
        ncol=3,
        fontsize=9
    )

    fig.subplots_adjust(bottom=0.17)

    plt.savefig(
        os.path.join(
            OUT_DIR, f"peta_prediksi_{kota}.png"
        ),
        dpi=300,
        bbox_inches="tight"
    )

    plt.close(fig)

print("Seluruh peta prediksi selesai disimpan:", OUT_DIR)


Seluruh peta prediksi selesai disimpan: /content/drive/MyDrive/SolarPanel_Training/Hasil_Luas_Area
